# EWC experiments on Colab GPU

Runs the conditions and seeds set at the top of `ewc/runEwcExperiments.py`, ANN and SNN.
Code is cloned from GitHub, so **commit and push local changes first**.

In [12]:
# 1. Confirm this kernel is the Colab T4, not your laptop.
import platform, torch
print("machine:", platform.node(), platform.system())
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: select the Colab kernel and a T4 GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

machine: 691402142ae6 Linux
torch: 2.14.1+cu130 | CUDA available: True
GPU: Tesla T4


In [13]:
# 2. Get the latest code from GitHub and install the one missing package.
import os, sys
REPO = "/content/Dissertation"
if not os.path.exists(REPO):
    !git clone -q https://github.com/ghonilcloud/Dissertation.git {REPO}
%cd {REPO}
!git pull -q
!git log --oneline -1
!pip install -q spikingjelly
sys.path.insert(0, REPO)

/content/Dissertation
b208f07 (HEAD -> main, origin/main, origin/HEAD) cond 3 for google colab file


In [ ]:
# 3. Stop early if GitHub still has the old code (conditions without ewc_lambda/replay).
import ewc.runEwcExperiments as r
assert "replay" in r.EWC_EXPERIMENT_CONDITIONS[0], "Old code on GitHub: commit and push, then re-run cell 2."
print(r.EWC_EXPERIMENT_CONDITIONS[0])

{'epochs': 1, 'time_steps': 2, 'ewc_lambda': 100.0, 'replay': False}


In [15]:
# 4. Download CIFAR-10 (data/ is not in git).
# Fast path: the archive attached to the repo's GitHub release (tag "cifar10").
# torchvision then finds it, checks its MD5, skips its own slow download and just extracts.
# If the release is missing, torchvision falls back to the (slow) Toronto server.
!mkdir -p data
!rm -f data/cifar-10-python.tar.gz  # remove any half-finished download
!wget -q --show-progress -O data/cifar-10-python.tar.gz https://github.com/ghonilcloud/Dissertation/releases/download/cifar10/cifar-10-python.tar.gz || rm -f data/cifar-10-python.tar.gz
from utilities.dataset import get_cifar_datasets
_ = get_cifar_datasets("cifar10", "data", download=True)

data/cifar-10-pytho 100%[===================>] 162.60M   273MB/s    in 0.6s    


In [ ]:
# 5. Run every active condition and seed set in ewc/runEwcExperiments.py
#    (edit EWC_EXPERIMENT_CONDITIONS and SEEDS there, then commit, push and re-run cell 2).
import importlib, time
importlib.reload(r)  # pick up the latest file after a git pull
OUTPUT = "results/task_ewc_replay.json"  # task mode, EWC (lambda=100) + replay (200/task)

print("conditions:", r.EWC_EXPERIMENT_CONDITIONS)
print("seeds:", r.SEEDS)
start = time.perf_counter()
res = r.run_grid(output_path=OUTPUT, progress=True, scenario="task")
wall = time.perf_counter() - start
for c in res["conditions"]:
    for m in ("ann", "snn"):
        print(f"{c['epochs']} epochs | {m}: final accuracy {c[m]['mean_final_accuracy']:.3f}, training time {c[m]['training_time_total_seconds']:.1f}s")
print(f"wall time (incl. EWC Fisher + evaluation): {wall:.1f}s")

In [ ]:
# 6. Save the result to your computer (Colab files are deleted when the session ends).
from google.colab import files
files.download(OUTPUT)